# Step 1 — Mount Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Step 2 — Find your GeoVision folder and see what's actually there

In [ ]:
import os

base = "/content/drive/MyDrive"

for item in os.listdir(base):
    if "GeoVision" in item or "geovision" in item.lower():
        print(item)

GeoVision exports (1)
GeoVision exports


# Search both GeoVision folders for tiles directories

In [ ]:
import os

search_roots = [
    "/content/drive/MyDrive/GeoVision exports",
    "/content/drive/MyDrive/GeoVision exports (1)",
]

for root in search_roots:
    print(f"\n=== Contents of: {root} ===")
    for item in os.listdir(root):
        print(" ", item)


=== Contents of: /content/drive/MyDrive/GeoVision exports ===
  city_islamabad.tif
  city_karachi.tif
  city_peshawar.tif
  city_faisalabad.tif
  city_lahore.tif
  city_karachi_fixed.tif
  city_karachi_v3.tif
  task_b

=== Contents of: /content/drive/MyDrive/GeoVision exports (1) ===
  sentinel2_pakistan-0000000000-0000000000.tif
  sentinel2_pakistan-0000000000-0000009472.tif
  sentinel2_pakistan-0000009472-0000000000.tif
  sentinel2_pakistan-0000009472-0000009472.tif
  worldcover_labels.tif
  tiles
  graveyards
  graveyards_yolo


# Search your entire Drive for any folder literally named "tiles" or

In [ ]:
for dirpath, dirnames, filenames in os.walk("/content/drive/MyDrive"):
    for d in dirnames:
        if d in ("tiles", "images", "masks"):
            full = os.path.join(dirpath, d)
            try:
                count = len(os.listdir(full))
            except:
                count = "?"
            print(f"{full}  ({count} items)")

/content/drive/MyDrive/GeoVision exports (1)/tiles  (2 items)
/content/drive/MyDrive/GeoVision exports (1)/tiles/images  (11306 items)
/content/drive/MyDrive/GeoVision exports (1)/tiles/masks  (11305 items)
/content/drive/MyDrive/GeoVision exports (1)/graveyards_yolo/images  (3 items)


# Check Task A tiles inside the correct folder

In [ ]:
import os

task_a_base = "/content/drive/MyDrive/GeoVision exports (1)/tiles"

print("Images:", len(os.listdir(f"{task_a_base}/images")))
print("Masks :", len(os.listdir(f"{task_a_base}/masks")))
print("Splits file exists:", os.path.exists(f"{task_a_base}/splits.json"))

Images: 11306
Masks : 11305
Splits file exists: False


# Check one tile's shape

In [ ]:
import numpy as np

sample_img = sorted(os.listdir(f"{task_a_base}/images"))[0]
img  = np.load(f"{task_a_base}/images/{sample_img}")
mask = np.load(f"{task_a_base}/masks/{sample_img}")

print("Sample file:", sample_img)
print("Image shape:", img.shape)
print("Mask shape :", mask.shape)
print("Mask unique values:", np.unique(mask))

Sample file: tile_000000.npy
Image shape: (6, 224, 224)
Mask shape : (224, 224)
Mask unique values: [0 1 2 3 4]


# Check Task C YOLO structure

In [ ]:
yolo_base = "/content/drive/MyDrive/GeoVision exports (1)/graveyards_yolo"

for split in ["train", "val", "test"]:
    n_img = len(os.listdir(f"{yolo_base}/images/{split}"))
    n_lbl = len(os.listdir(f"{yolo_base}/labels/{split}"))
    print(f"{split}: {n_img} images, {n_lbl} labels")

print("dataset.yaml exists:", os.path.exists(f"{yolo_base}/dataset.yaml"))

train: 2349 images, 2349 labels
val: 335 images, 335 labels
test: 672 images, 672 labels
dataset.yaml exists: True


# Print dataset.yaml contents to verify paths inside it are correct

In [ ]:
with open(f"{yolo_base}/dataset.yaml") as f:
    print(f.read())


path: /content/drive/MyDrive/GeoVision exports/graveyards_yolo
train: images/train
val:   images/val
test:  images/test

nc: 1
names: ['graveyard']



# Issue 1 — splits.json is missing for Task A

In [ ]:
import os, json, random

task_a_base = "/content/drive/MyDrive/GeoVision exports (1)/tiles"

all_tiles = sorted([
    f"{task_a_base}/images/{f}"
    for f in os.listdir(f"{task_a_base}/images")
])

random.seed(42)
random.shuffle(all_tiles)

n         = len(all_tiles)
train_end = int(0.70 * n)
val_end   = int(0.80 * n)

splits = {
    "train": all_tiles[:train_end],
    "val":   all_tiles[train_end:val_end],
    "test":  all_tiles[val_end:]
}

with open(f"{task_a_base}/splits.json", "w") as f:
    json.dump(splits, f)

print(f"Train: {len(splits['train'])}")
print(f"Val  : {len(splits['val'])}")
print(f"Test : {len(splits['test'])}")

Train: 7914
Val  : 1130
Test : 2262


# Issue 1 — the 1 orphan image with no mask

In [ ]:
img_files  = set(os.listdir(f"{task_a_base}/images"))
mask_files = set(os.listdir(f"{task_a_base}/masks"))

orphans = img_files - mask_files
print("Orphan image(s) with no mask:", orphans)

Orphan image(s) with no mask: {'tile_011305.npy'}


In [ ]:
for orphan in orphans:
    os.remove(f"{task_a_base}/images/{orphan}")
    print(f"Removed: {orphan}")

Removed: tile_011305.npy


# Issue 2 — splits.json is missing for Task A

In [ ]:
import os, json, random

task_a_base = "/content/drive/MyDrive/GeoVision exports (1)/tiles"

all_tiles = sorted([
    f"{task_a_base}/images/{f}"
    for f in os.listdir(f"{task_a_base}/images")
])

random.seed(42)
random.shuffle(all_tiles)

n         = len(all_tiles)
train_end = int(0.70 * n)
val_end   = int(0.80 * n)

splits = {
    "train": all_tiles[:train_end],
    "val":   all_tiles[train_end:val_end],
    "test":  all_tiles[val_end:]
}

with open(f"{task_a_base}/splits.json", "w") as f:
    json.dump(splits, f)

print(f"Train: {len(splits['train'])}")
print(f"Val  : {len(splits['val'])}")
print(f"Test : {len(splits['test'])}")

Train: 7913
Val  : 1131
Test : 2261


# Issue 3 — stale path in dataset.yaml

In [ ]:
yolo_base = "/content/drive/MyDrive/GeoVision exports (1)/graveyards_yolo"

yaml_content = f"""path: {yolo_base}
train: images/train
val: images/val
test: images/test

nc: 1
names: ['graveyard']
"""

with open(f"{yolo_base}/dataset.yaml", "w") as f:
    f.write(yaml_content)

print(yaml_content)

path: /content/drive/MyDrive/GeoVision exports (1)/graveyards_yolo
train: images/train
val: images/val
test: images/test

nc: 1
names: ['graveyard']



# New Section

In [ ]:
import numpy as np

task_a_base = "/content/drive/MyDrive/GeoVision exports (1)/tiles"
sample = np.load(f"{task_a_base}/images/tile_000000.npy")

print("Shape:", sample.shape)
print("Dtype:", sample.dtype)
print("Min  :", sample.min())
print("Max  :", sample.max())
print("Mean per band:", sample.mean(axis=(1,2)))

Shape: (6, 224, 224)
Dtype: float64
Min  : nan
Max  : nan
Mean per band: [nan nan nan nan nan nan]


In [ ]:
import shutil, os, time

local_dir = "/content/sample_check"
os.makedirs(local_dir, exist_ok=True)

sample_files = sorted(os.listdir(f"{task_a_base}/images"))[:50]  # just 50, not 200

start = time.time()
for fname in sample_files:
    shutil.copy(f"{task_a_base}/images/{fname}", f"{local_dir}/{fname}")
print(f"Copied 50 files in {time.time()-start:.1f}s")